# TravelSalesAgent 評価ノートブック

このノートブックは [Microsoft Learn: データ エージェントを評価する](https://learn.microsoft.com/ja-jp/fabric/data-science/evaluate-data-agent) に基づき、`TravelSalesAgent` を Fabric Data Agent SDK でプログラム評価します。

## 実行前の準備

1. このノートブックを **TravelSalesLakehouse が存在するワークスペース** の Fabric Notebook としてインポート
2. ノートブックに **TravelSalesLakehouse を Lakehouse として追加**（左パネル → 「Lakehouse の追加」）
3. `evaluation/evaluation_questions.csv` を Lakehouse の `Files/data/` フォルダにアップロード
   - Lakehouse 画面 → Files → data フォルダ → ファイルをアップロード

> ⚠️ Data Agent の評価機能は **F2 以上の有償 Fabric 容量** が必要です。

## Step 1: Fabric Data Agent SDK をインストールする

In [ ]:
%pip install -U fabric-data-agent-sdk

## Step 2: グラウンド トゥルース データセットを読み込む

`evaluation_questions.csv` には、ハンズオン Step 4 の質問 1〜5 と、CSV データから集計した正解が含まれています。

In [ ]:
import pandas as pd

# Lakehouse の Files/data/ にアップロードした CSV を読み込む（フォルダ名は小文字の "data"）
input_file_path = "/lakehouse/default/Files/data/evaluation_questions.csv"
df = pd.read_csv(input_file_path)

print(f"評価データ件数: {len(df)} 件")
df

## Step 3: Data Agent を評価する

`evaluate_data_agent` 関数が各質問を `TravelSalesAgent` に送信し、実際の回答と期待回答を LLM（critic）で比較します。

- 結果は Lakehouse のデルタテーブル `travel_evaluation_output` および `travel_evaluation_output_steps` に保存されます。
- `evaluation_id` を使って後から詳細を参照できます。

In [ ]:
from fabric.dataagent.evaluation import evaluate_data_agent

# ハンズオンで作成した Data Agent の名前
data_agent_name = "TravelSalesAgent"

# Data Agent が同じワークスペースにある場合は None のまま
workspace_name = None

# 評価結果を保存するテーブル名（Lakehouse に作成される）
table_name = "travel_evaluation_output"

# 評価を実行
evaluation_id = evaluate_data_agent(
    df,
    data_agent_name,
    workspace_name=workspace_name,
    table_name=table_name,
    data_agent_stage="production",
)

print(f"評価実行 ID: {evaluation_id}")

## Step 4: 評価サマリーを確認する

正答数・誤答数・不明数と精度（Accuracy）を確認します。

In [ ]:
from fabric.dataagent.evaluation import get_evaluation_summary

summary_df = get_evaluation_summary(table_name=table_name, verbose=True)
summary_df

## Step 5: 質問ごとの詳細結果を確認する

各質問の実際の回答（`actual_answer`）と評価結果（`true` / `false` / `unclear`）を確認します。

- `get_all_rows=True` にするとすべての質問の結果を表示します
- `get_all_rows=False`（デフォルト）では不正解・不明の質問のみを表示します

In [ ]:
from fabric.dataagent.evaluation import get_evaluation_details

detail_df = get_evaluation_details(
    evaluation_id,
    table_name=table_name,
    get_all_rows=True,   # すべての質問結果を表示
    verbose=True,
)
detail_df